# P100 GPU core-path validation: torch 2.0.1+cu117 + SadTalker + SDXL-Turbo

In [ ]:
# 0. Install P100-compatible PyTorch (cu117 supports sm_60) + SadTalker deps
import subprocess, sys, os, json
def run(cmd):
    print('$', ' '.join(cmd[:4]))
    r=subprocess.run(cmd,check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    return r.returncode
# 1) P100-capable torch
run([sys.executable,'-m','pip','install','-q','--no-cache-dir','torch==2.0.1+cu117','torchvision==0.15.2+cu117','--index-url','https://download.pytorch.org/whl/cu117'])
# 2) numpy/skimage pinned + force rebuild against this numpy
for p in ['numpy==1.26.4','scipy==1.13.1','scikit-image==0.22.0','imageio==2.34.0','imageio-ffmpeg==0.5.1']:
    run([sys.executable,'-m','pip','install','-q',p])
run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps','scikit-image==0.22.0','scipy==1.13.1'])
# 3) SadTalker deps
for p in ['kornia==0.7.2','dlib','face_alignment==1.3.5','basicsr==1.4.2','facexlib==0.3.0','gfpgan','av','safetensors','yacs==0.1.8','einops','opencv-python-headless','librosa==0.10.2','numba==0.60.0','xformers==0.0.22']:
    run([sys.executable,'-m','pip','install','-q',p])
# 4) re-pin torch so nothing upgraded it
run([sys.executable,'-m','pip','install','-q','--no-cache-dir','torch==2.0.1+cu117','torchvision==0.15.2+cu117','--index-url','https://download.pytorch.org/whl/cu117'])
import torch, numpy, skimage
print('torch', torch.__version__, '| cuda avail', torch.cuda.is_available(), '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('numpy', numpy.__version__, '| skimage', skimage.__version__)
if torch.cuda.is_available():
    t=torch.zeros(3,3,device='cuda'); print('cuda tensor ok', t.device)
    print('arch list supported by this torch build includes sm_60?', 'sm_60' in torch.cuda.get_arch_list())
print('install cell done')


In [ ]:
# 1. Profile + 30s Kokoro voice (CPU)
import subprocess, sys, os
subprocess.run([sys.executable,'-m','pip','install','-q','gdown','kokoro','soundfile'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
subprocess.run(['apt-get','-qq','install','-y','ffmpeg'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
os.makedirs('/kaggle/working/test',exist_ok=True)
subprocess.run(['gdown','1-2sFUEHqXDbaPq0lfBmamrjQBsdL_QuY','-O','/kaggle/working/test/profile.jpg'],check=False)
print('profile', os.path.getsize('/kaggle/working/test/profile.jpg'))
from kokoro import KPipeline
import soundfile as sf, numpy as np
pipe=KPipeline(lang_code='a')
txt='My name is Mirsina Aghdam, CEO of EDGE, Earthwise Dynamics Geo Environs. We are an Irish company working in geoengineering, AI automation and critical-mineral intelligence. Europe needs secure rare-earth supplies, but exploration remains slow and fragmented.'
chunks=[]
for gs,ps,a in pipe(txt, voice='am_michael', speed=1.0):
    chunks.append(a.cpu().numpy())
a=np.concatenate(chunks) if chunks else np.zeros(24000*30)
a=a[:24000*30] if len(a)>=24000*30 else np.pad(a,(0,24000*30-len(a)))
sf.write('/kaggle/working/test/seg.wav', a.astype('float32'), 24000)
print('audio written', os.path.getsize('/kaggle/working/test/seg.wav'))


In [ ]:
# 2. Clone SadTalker + download models
import os, subprocess
if not os.path.exists('/kaggle/working/SadTalker'):
    subprocess.run(['git','clone','https://github.com/OpenTalker/SadTalker.git','/kaggle/working/SadTalker'],check=True)
os.chdir('/kaggle/working/SadTalker')
subprocess.run(['bash','scripts/download_models.sh'],check=True)
print('models:', os.path.exists('checkpoints/SadTalker_V0.0.2_512.safetensors'))


In [ ]:
# 3. Run SadTalker ONE 30s clip
import os, subprocess, glob
os.chdir('/kaggle/working/SadTalker')
cmd=['python','inference.py','--driven_audio','/kaggle/working/test/seg.wav','--source_image','/kaggle/working/test/profile.jpg','--result_dir','/kaggle/working/sad_test','--size','512','--preprocess','crop','--still','--enhancer','gfpgan','--batch_size','1']
print('running sadtalker...'); r=subprocess.run(cmd,check=False)
print('returncode', r.returncode)
files=glob.glob('/kaggle/working/sad_test/*/*.mp4')
print('output files', files)
if files:
    import shutil
    shutil.copy(files[0],'/kaggle/working/test/sad_clip.mp4')
    print('copied', files[0])


In [ ]:
# 4. SDXL-Turbo smoke test (1 image)
import subprocess, sys, torch
subprocess.run([sys.executable,'-m','pip','install','-q','diffusers','transformers','accelerate'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
from diffusers import AutoPipelineForText2Image
pipe=AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16').to('cuda')
im=pipe('Photorealistic scientific image of Earth viewed from low orbit, realistic blue atmosphere, dark space, small satellite, no text', num_inference_steps=1, guidance_scale=0.0).images[0]
im.save('/kaggle/working/test/sdxl_test.png')
print('sdxl image saved', im.size)


In [ ]:
# 5. Report
import os, glob
print('SADTALKER:', glob.glob('/kaggle/working/test/sad_clip.mp4'))
print('SDXL:', os.path.exists('/kaggle/working/test/sdxl_test.png'))
print('AUDIO:', os.path.exists('/kaggle/working/test/seg.wav'), os.path.getsize('/kaggle/working/test/seg.wav') if os.path.exists('/kaggle/working/test/seg.wav') else 0)
print('PROFILE:', os.path.exists('/kaggle/working/test/profile.jpg'))
